In [2]:
from IPython.display import display
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.stats import norm
from datetime import datetime, timedelta

ticker = "UNH"
expiration = "2026-04-24"
underline = 324.63
risk_free_rate = 0.045
contract_size = 100

ticker_obj = yf.Ticker(ticker)
chain = ticker_obj.option_chain(expiration)

calls = chain.calls[["strike", "bid", "ask", "impliedVolatility"]].copy()
puts = chain.puts[["strike", "bid", "ask", "impliedVolatility"]].copy()

calls["optionType"] = "call"
puts["optionType"] = "put"

for df in [calls, puts]:
    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["impliedVolatility"] = pd.to_numeric(df["impliedVolatility"], errors="coerce")

underlying_price = float(underline)

expiry_dt = datetime.strptime(expiration, "%Y-%m-%d") + timedelta(hours=16)
now_dt = datetime.now()
T = max((expiry_dt - now_dt).total_seconds(), 60) / (365 * 24 * 60 * 60)

def bs_delta(S, K, T, r, sigma, option_type):
    if pd.isna(S) or pd.isna(K) or pd.isna(T) or pd.isna(r) or pd.isna(sigma):
        return np.nan
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        return np.nan

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

    if option_type == "call":
        return norm.cdf(d1)
    elif option_type == "put":
        return norm.cdf(d1) - 1
    else:
        return np.nan

calls["delta"] = calls.apply(
    lambda row: bs_delta(
        S=underlying_price,
        K=row["strike"],
        T=T,
        r=risk_free_rate,
        sigma=row["impliedVolatility"],
        option_type="call"
    ),
    axis=1
)

puts["delta"] = puts.apply(
    lambda row: bs_delta(
        S=underlying_price,
        K=row["strike"],
        T=T,
        r=risk_free_rate,
        sigma=row["impliedVolatility"],
        option_type="put"
    ),
    axis=1
)

def color_pnl(val):
    if pd.isna(val):
        return ""
    if val > 0:
        return "color: green"
    if val < 0:
        return "color: red"
    return "color: black"

number_format = {
    "strike": "{:.2f}",
    "entry_premium": "{:.2f}",
    "mark_price": "{:.2f}",
    "underlying": "{:.2f}",
    "mtm_p_l": "{:,.0f}",
    "exp_p_l": "{:,.0f}",
    "IV": "{:.2%}",
    "delta": "{:.3f}"
}

long_call_table = calls[["strike", "ask", "bid", "impliedVolatility", "delta"]].copy()
long_call_table["entry_premium"] = long_call_table["ask"]
long_call_table["mark_price"] = long_call_table["bid"]
long_call_table["underlying"] = underlying_price
long_call_table["mtm_p_l"] = (long_call_table["mark_price"] - long_call_table["entry_premium"]) * contract_size
long_call_table["exp_p_l"] = (
    np.maximum(long_call_table["underlying"] - long_call_table["strike"], 0)
    - long_call_table["entry_premium"]
) * contract_size
long_call_table["IV"] = long_call_table["impliedVolatility"]
long_call_table = long_call_table[["strike", "entry_premium", "mark_price", "underlying", "mtm_p_l", "exp_p_l", "IV", "delta"]]

short_call_table = calls[["strike", "bid", "ask", "impliedVolatility", "delta"]].copy()
short_call_table["entry_premium"] = short_call_table["bid"]
short_call_table["mark_price"] = short_call_table["ask"]
short_call_table["underlying"] = underlying_price
short_call_table["mtm_p_l"] = (short_call_table["entry_premium"] - short_call_table["mark_price"]) * contract_size
short_call_table["exp_p_l"] = (
    short_call_table["entry_premium"]
    - np.maximum(short_call_table["underlying"] - short_call_table["strike"], 0)
) * contract_size
short_call_table["IV"] = short_call_table["impliedVolatility"]
short_call_table = short_call_table[["strike", "entry_premium", "mark_price", "underlying", "mtm_p_l", "exp_p_l", "IV", "delta"]]

long_put_table = puts[["strike", "ask", "bid", "impliedVolatility", "delta"]].copy()
long_put_table["entry_premium"] = long_put_table["ask"]
long_put_table["mark_price"] = long_put_table["bid"]
long_put_table["underlying"] = underlying_price
long_put_table["mtm_p_l"] = (long_put_table["mark_price"] - long_put_table["entry_premium"]) * contract_size
long_put_table["exp_p_l"] = (
    np.maximum(long_put_table["strike"] - long_put_table["underlying"], 0)
    - long_put_table["entry_premium"]
) * contract_size
long_put_table["IV"] = long_put_table["impliedVolatility"]
long_put_table = long_put_table[["strike", "entry_premium", "mark_price", "underlying", "mtm_p_l", "exp_p_l", "IV", "delta"]]

short_put_table = puts[["strike", "bid", "ask", "impliedVolatility", "delta"]].copy()
short_put_table["entry_premium"] = short_put_table["bid"]
short_put_table["mark_price"] = short_put_table["ask"]
short_put_table["underlying"] = underlying_price
short_put_table["mtm_p_l"] = (short_put_table["entry_premium"] - short_put_table["mark_price"]) * contract_size
short_put_table["exp_p_l"] = (
    short_put_table["entry_premium"]
    - np.maximum(short_put_table["strike"] - short_put_table["underlying"], 0)
) * contract_size
short_put_table["IV"] = short_put_table["impliedVolatility"]
short_put_table = short_put_table[["strike", "entry_premium", "mark_price", "underlying", "mtm_p_l", "exp_p_l", "IV", "delta"]]

print("LONG CALL")
display(
    long_call_table.style
    .map(color_pnl, subset=["mtm_p_l", "exp_p_l"])
    .format(number_format)
)

print("SHORT CALL")
display(
    short_call_table.style
    .map(color_pnl, subset=["mtm_p_l", "exp_p_l"])
    .format(number_format)
)

print("LONG PUT")
display(
    long_put_table.style
    .map(color_pnl, subset=["mtm_p_l", "exp_p_l"])
    .format(number_format)
)

print("SHORT PUT")
display(
    short_put_table.style
    .map(color_pnl, subset=["mtm_p_l", "exp_p_l"])
    .format(number_format)
)

LONG CALL


,strike,entry_premium,mark_price,underlying,mtm_p_l,exp_p_l,IV,delta
0,145.00,182.30,176.05,324.63,-625,-267,369.43%,0.974
1,150.00,177.30,171.05,324.63,-625,-267,355.76%,0.973
2,155.00,172.30,166.05,324.63,-625,-267,342.53%,0.972
3,160.00,167.30,161.05,324.63,-625,-267,329.69%,0.970
4,170.00,157.30,151.05,324.63,-625,-267,305.22%,0.968
5,175.00,152.30,146.05,324.63,-625,-267,293.46%,0.967
6,200.00,127.35,121.10,324.63,-625,-272,240.28%,0.958
7,215.00,112.40,106.15,324.63,-625,-277,211.43%,0.951
8,220.00,106.80,101.15,324.63,-565,-217,190.41%,0.957
9,225.00,102.45,96.20,324.63,-625,-282,193.31%,0.946


SHORT CALL


,strike,entry_premium,mark_price,underlying,mtm_p_l,exp_p_l,IV,delta
0,145.00,176.05,182.30,324.63,-625,-358,369.43%,0.974
1,150.00,171.05,177.30,324.63,-625,-358,355.76%,0.973
2,155.00,166.05,172.30,324.63,-625,-358,342.53%,0.972
3,160.00,161.05,167.30,324.63,-625,-358,329.69%,0.970
4,170.00,151.05,157.30,324.63,-625,-358,305.22%,0.968
5,175.00,146.05,152.30,324.63,-625,-358,293.46%,0.967
6,200.00,121.10,127.35,324.63,-625,-353,240.28%,0.958
7,215.00,106.15,112.40,324.63,-625,-348,211.43%,0.951
8,220.00,101.15,106.80,324.63,-565,-348,190.41%,0.957
9,225.00,96.20,102.45,324.63,-625,-343,193.31%,0.946


LONG PUT


,strike,entry_premium,mark_price,underlying,mtm_p_l,exp_p_l,IV,delta
0,145.00,0.02,0.00,324.63,-2,-2,181.25%,-0.000
1,150.00,0.02,0.00,324.63,-2,-2,175.00%,-0.000
2,155.00,0.02,0.00,324.63,-2,-2,167.19%,-0.000
3,160.00,0.02,0.00,324.63,-2,-2,159.38%,-0.000
4,175.00,0.02,0.01,324.63,-1,-2,145.31%,-0.000
5,180.00,0.03,0.01,324.63,-2,-3,143.75%,-0.000
6,185.00,0.03,0.01,324.63,-2,-3,137.50%,-0.001
7,190.00,0.03,0.01,324.63,-2,-3,131.25%,-0.001
8,195.00,0.03,0.01,324.63,-2,-3,125.00%,-0.001
9,200.00,0.07,0.01,324.63,-6,-7,127.34%,-0.001


SHORT PUT


,strike,entry_premium,mark_price,underlying,mtm_p_l,exp_p_l,IV,delta
0,145.00,0.00,0.02,324.63,-2,0,181.25%,-0.000
1,150.00,0.00,0.02,324.63,-2,0,175.00%,-0.000
2,155.00,0.00,0.02,324.63,-2,0,167.19%,-0.000
3,160.00,0.00,0.02,324.63,-2,0,159.38%,-0.000
4,175.00,0.01,0.02,324.63,-1,1,145.31%,-0.000
5,180.00,0.01,0.03,324.63,-2,1,143.75%,-0.000
6,185.00,0.01,0.03,324.63,-2,1,137.50%,-0.001
7,190.00,0.01,0.03,324.63,-2,1,131.25%,-0.001
8,195.00,0.01,0.03,324.63,-2,1,125.00%,-0.001
9,200.00,0.01,0.07,324.63,-6,1,127.34%,-0.001
